
# NSLAB Logistics Intelligence Platform
## 01 — Inspeção Quantitativa: LaDe-P Shanghai

**Dataset:** LaDe-P (Pickup) — Shanghai  
**Origem:** Cainiao-AI / Hugging Face  
**Formato:** Apache Parquet

### Objetivo

Investigar a estrutura, a qualidade e as características operacionais do dataset antes de iniciar o ETL.

### Escopo da inspeção

1. Metadados e estrutura do Parquet
2. Valores ausentes e duplicatas
3. Identificadores e relacionamentos
4. Datas e períodos de coleta
5. SLA e distribuição dos atrasos
6. Variáveis candidatas para Machine Learning
7. Riscos de data leakage
8. Coordenadas geográficas e eventos operacionais

### Premissas

- Preservar integralmente os dados originais.
- Priorizar inspeções eficientes em memória.
- Não realizar transformações ETL nesta etapa.
- Documentar as descobertas com base nos dados reais.


In [15]:

# 1. Inspeção dos metadados do Parquet
# Leitura sem carregar os registros na memória

from pathlib import Path
import pyarrow.parquet as pq

# Caminho relativo à pasta notebooks
arquivo = (
    Path("..")
    / "data"
    / "raw"
    / "pickup"
    / "pickup_sh-00000-of-00001-79fabe8088e723a2.parquet"
)

if not arquivo.is_file():
    raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")

# Acessar somente os metadados
parquet = pq.ParquetFile(arquivo)
metadata = parquet.metadata
schema = parquet.schema_arrow

print("=== METADADOS DO PARQUET ===")
print(f"Arquivo: {arquivo.name}")
print(f"Tamanho (MiB): {arquivo.stat().st_size / 1024**2:.2f}")
print(f"Registros: {metadata.num_rows:,}")
print(f"Colunas: {metadata.num_columns}")
print(f"Row groups: {metadata.num_row_groups}")

print("\n=== COLUNAS E TIPOS ===")
for campo in schema:
    print(f"{campo.name}: {campo.type}")


=== METADADOS DO PARQUET ===
Arquivo: pickup_sh-00000-of-00001-79fabe8088e723a2.parquet
Tamanho (MiB): 93.64
Registros: 1,424,406
Colunas: 19
Row groups: 1425

=== COLUNAS E TIPOS ===
order_id: int64
region_id: int64
city: string
courier_id: int64
accept_time: string
time_window_start: string
time_window_end: string
lng: double
lat: double
aoi_id: int64
aoi_type: int64
pickup_time: string
pickup_gps_time: string
pickup_gps_lng: double
pickup_gps_lat: double
accept_gps_time: string
accept_gps_lng: double
accept_gps_lat: double
ds: int64


In [16]:
# 2. Distribuição dos registros por row group
# Utiliza somente os metadados já carregados

registros_por_grupo = [
    metadata.row_group(i).num_rows
    for i in range(metadata.num_row_groups)
]

print("=== DISTRIBUIÇÃO DOS ROW GROUPS ===")
print(f"Quantidade: {len(registros_por_grupo):,}")
print(f"Menor grupo: {min(registros_por_grupo):,} registros")
print(f"Maior grupo: {max(registros_por_grupo):,} registros")
print(
    f"Média: "
    f"{sum(registros_por_grupo) / len(registros_por_grupo):,.2f}"
)
print(
    f"Soma dos registros: "
    f"{sum(registros_por_grupo):,}"
)

print(
    "Consistência com metadados:",
    sum(registros_por_grupo) == metadata.num_rows
)


=== DISTRIBUIÇÃO DOS ROW GROUPS ===
Quantidade: 1,425
Menor grupo: 406 registros
Maior grupo: 1,000 registros
Média: 999.58
Soma dos registros: 1,424,406
Consistência com metadados: True



## 2. Estrutura interna do Apache Parquet

O Apache Parquet é um formato de armazenamento colunar,
otimizado para processamento analítico de dados.

Diferentemente do CSV, que organiza os dados por linhas,
o Parquet organiza os registros em grupos de linhas
(*row groups*) e armazena suas colunas separadamente.

### 2.1 Componentes principais

- **Row group:** conjunto de registros dentro do arquivo.
- **Column chunk:** dados de uma coluna dentro de um row group.
- **Page:** subdivisão de um column chunk, utilizada para
  codificação e compressão dos dados.
- **Schema:** definição das colunas e seus tipos.
- **Footer:** região final do arquivo que contém metadados,
  incluindo schema e localização dos grupos de dados.

### 2.2 Vantagens do formato

- Leitura seletiva de colunas.
- Compressão eficiente.
- Redução de operações de entrada e saída (I/O).
- Possibilidade de leitura por lotes.
- Integração com PyArrow, Pandas, PySpark e Databricks.

### 2.3 Estrutura identificada no LaDe-P Shanghai

| Propriedade | Resultado |
|---|---:|
| Registros | 1.424.406 |
| Colunas | 19 |
| Tamanho | 93,64 MiB |
| Row groups | 1.425 |
| Menor row group | 406 registros |
| Maior row group | 1.000 registros |
| Média por grupo | 999,58 registros |

A soma dos registros dos row groups corresponde ao total
informado nos metadados do arquivo.

### 2.4 PyArrow e inspeção de metadados

A biblioteca `pyarrow.parquet` permite consultar a
estrutura física do Parquet sem carregar todos os registros.

Principais recursos utilizados:

| Recurso | Finalidade |
|---|---|
| `pq.ParquetFile()` | Abrir o arquivo para leitura |
| `.metadata` | Consultar metadados |
| `.schema_arrow` | Consultar colunas e tipos |
| `.num_rows` | Consultar quantidade de registros |
| `.num_row_groups` | Consultar quantidade de grupos |
| `.row_group(i)` | Consultar metadados de um grupo |

### 2.5 Próxima verificação

Inspecionar o primeiro row group para identificar:

- Quantidade de registros e colunas.
- Algoritmos de compressão utilizados.
- Tamanho comprimido das colunas.
- Organização dos column chunks.

Essa inspeção utilizará somente metadados,
preservando integralmente o dataset original.


In [17]:

# 3. Inspeção do primeiro row group
# Consulta somente aos metadados do Parquet

grupo = metadata.row_group(0)

print("=== ROW GROUP 0 ===")
print(f"Registros: {grupo.num_rows:,}")
print(f"Colunas: {grupo.num_columns}")
print(f"Tamanho não comprimido: {grupo.total_byte_size:,} bytes")

print("\n=== COLUMN CHUNKS ===")

for i in range(grupo.num_columns):
    coluna = grupo.column(i)

    print(
        f"{coluna.path_in_schema:25} "
        f"Compressão: {coluna.compression:10} "
        f"Tamanho comprimido: {coluna.total_compressed_size:,} bytes"
    )


=== ROW GROUP 0 ===
Registros: 1,000
Colunas: 19
Tamanho não comprimido: 131,244 bytes

=== COLUMN CHUNKS ===
order_id                  Compressão: SNAPPY     Tamanho comprimido: 6,505 bytes
region_id                 Compressão: SNAPPY     Tamanho comprimido: 99 bytes
city                      Compressão: SNAPPY     Tamanho comprimido: 83 bytes
courier_id                Compressão: SNAPPY     Tamanho comprimido: 544 bytes
accept_time               Compressão: SNAPPY     Tamanho comprimido: 8,118 bytes
time_window_start         Compressão: SNAPPY     Tamanho comprimido: 4,470 bytes
time_window_end           Compressão: SNAPPY     Tamanho comprimido: 4,436 bytes
lng                       Compressão: SNAPPY     Tamanho comprimido: 5,214 bytes
lat                       Compressão: SNAPPY     Tamanho comprimido: 4,968 bytes
aoi_id                    Compressão: SNAPPY     Tamanho comprimido: 322 bytes
aoi_type                  Compressão: SNAPPY     Tamanho comprimido: 127 bytes
pickup_time

### Entendendo o código acima
- Expressão	Significado
- metadata.row_group(0)	Seleciona os metadados do primeiro grupo
- grupo.num_rows	Quantidade de registros do grupo
- grupo.num_columns	Quantidade de colunas
- grupo.total_byte_size	Tamanho não comprimido informado nos metadados
- grupo.column(i)	Seleciona os metadados de uma coluna
- coluna.path_in_schema	Nome/caminho da coluna
- coluna.compression	Algoritmo de compressão
- coluna.total_compressed_size	Tamanho comprimido do column chunk

## Passo 6 — Verificação de valores ausentes

Agora vamos identificar quantos valores nulos existem em cada uma das 19 colunas do dataset Shanghai.

Utilizaremos o PyArrow com leitura em lotes, evitando carregar os 1.424.406 registros simultaneamente na memória.

### 1. Conceitos importantes

- NULL: ausência de valor registrada no formato de dados.
- `null_count`: propriedade do PyArrow que retorna a quantidade de valores nulos em um array.
- `iter_batches()`: método que permite percorrer os registros em lotes (RecordBatches).
- `batch_size`: quantidade máxima de registros solicitada por lote.

Nesta primeira verificação, vamos contar apenas os nulos reconhecidos pelo PyArrow. Strings vazias, espaços em branco e outros marcadores de ausência serão investigados separadamente.

### 2. Código Python

No notebook `01_inspecao_shanghai.ipynb`, adicione uma nova célula Code:

```


In [18]:
# 4. Inspeção de valores ausentes
# Leitura em lotes, sem alterar os dados originais

from collections import defaultdict

nulos = defaultdict(int)
total_registros = 0

for lote in parquet.iter_batches(batch_size=10_000):
    total_registros += lote.num_rows

    for nome, coluna in zip(lote.schema.names, lote.columns):
        nulos[nome] += coluna.null_count

print("=== VALORES AUSENTES POR COLUNA ===")

for nome in parquet.schema_arrow.names:
    quantidade = nulos[nome]
    percentual = (quantidade / total_registros) * 100

    print(
        f"{nome:25} "
        f"Nulos: {quantidade:>10,} "
        f"Percentual: {percentual:>7.2f}%"
    )

print(f"\nTotal de registros analisados: {total_registros:,}")
print(
    "Consistência com metadados:",
    total_registros == metadata.num_rows
)


=== VALORES AUSENTES POR COLUNA ===
order_id                  Nulos:          0 Percentual:    0.00%
region_id                 Nulos:          0 Percentual:    0.00%
city                      Nulos:          0 Percentual:    0.00%
courier_id                Nulos:          0 Percentual:    0.00%
accept_time               Nulos:          0 Percentual:    0.00%
time_window_start         Nulos:          0 Percentual:    0.00%
time_window_end           Nulos:          0 Percentual:    0.00%
lng                       Nulos:          0 Percentual:    0.00%
lat                       Nulos:          0 Percentual:    0.00%
aoi_id                    Nulos:          0 Percentual:    0.00%
aoi_type                  Nulos:          0 Percentual:    0.00%
pickup_time               Nulos:          0 Percentual:    0.00%
pickup_gps_time           Nulos:    441,333 Percentual:   30.98%
pickup_gps_lng            Nulos:    441,333 Percentual:   30.98%
pickup_gps_lat            Nulos:    441,333 Percentual

### 3. Como funciona?

O processamento segue este fluxo:

Parquet — 1.424.406 registros

`iter_batches(batch_size=10_000)`

Lê lotes sucessivos de até 10.000 registros

`coluna.null_count`

Conta os nulos de cada coluna no lote

Resultado consolidado

19 contadores e percentuais de ausência

O `defaultdict(int)` mantém os contadores acumulados. Ao final, teremos a quantidade de nulos por coluna no arquivo inteiro, sem manter todos os lotes em memória.


### Consistência dos eventos GPS

Antes de avançarmos para duplicatas, recomendo verificar se os três campos de cada evento GPS estão ausentes simultaneamente.

Essa é uma verificação de qualidade importante, pois evita assumir que a ausência de horário implica necessariamente ausência das coordenadas.

### O que estamos verificando?

- Todos nulos: horário e ambas as coordenadas estão ausentes.
- Parcialmente nulos: pelo menos um campo está ausente, mas não todos.
- Todos preenchidos: os três campos possuem valores.

Essa inspeção continua utilizando PyArrow em lotes e não modifica os dados.

Execute apenas essa célula e me envie os resultados. Se os eventos GPS forem consistentes, documentaremos essa descoberta antes de avançar para a inspeção de duplicatas.

In [19]:

# 5. Consistência dos valores nulos nos eventos GPS
# Leitura em lotes, sem modificar os dados

import pyarrow.compute as pc

eventos = {
    "pickup_gps": [
        "pickup_gps_time",
        "pickup_gps_lng",
        "pickup_gps_lat",
    ],
    "accept_gps": [
        "accept_gps_time",
        "accept_gps_lng",
        "accept_gps_lat",
    ],
}

for evento, colunas in eventos.items():
    todos_nulos = 0
    parcialmente_nulos = 0
    todos_preenchidos = 0

    for lote in parquet.iter_batches(
        batch_size=10_000,
        columns=colunas
    ):
        mascaras = [
            pc.is_null(lote.column(i))
            for i in range(len(colunas))
        ]

        todos = pc.and_(
            pc.and_(mascaras[0], mascaras[1]),
            mascaras[2]
        )

        algum = pc.or_(
            pc.or_(mascaras[0], mascaras[1]),
            mascaras[2]
        )

        qtd_todos = pc.sum(todos).as_py()
        qtd_algum = pc.sum(algum).as_py()

        todos_nulos += qtd_todos
        parcialmente_nulos += qtd_algum - qtd_todos
        todos_preenchidos += lote.num_rows - qtd_algum

    print(f"\n=== {evento.upper()} ===")
    print(f"Todos nulos: {todos_nulos:,}")
    print(f"Parcialmente nulos: {parcialmente_nulos:,}")
    print(f"Todos preenchidos: {todos_preenchidos:,}")
    print(
        "Total consistente:",
        todos_nulos + parcialmente_nulos
        + todos_preenchidos == metadata.num_rows
    )



=== PICKUP_GPS ===
Todos nulos: 441,333
Parcialmente nulos: 0
Todos preenchidos: 983,073
Total consistente: True

=== ACCEPT_GPS ===
Todos nulos: 654,425
Parcialmente nulos: 0
Todos preenchidos: 769,981
Total consistente: True


## Passo 7 — Duplicatas e unicidade de order_id

### Objetivo

Verificar se cada `order_id` identifica exclusivamente uma coleta.

Serão analisados:
- Total de registros.
- Quantidade de identificadores distintos.
- Identificadores nulos.
- Identificadores que aparecem mais de uma vez.
- Quantidade de ocorrências duplicadas.

### Estratégia

Utilizar PyArrow para ler somente a coluna `order_id`,
em lotes de 10.000 registros.

As frequências dos identificadores serão acumuladas
em memória, sem carregar as demais colunas do dataset.

Nenhum registro será alterado ou excluído.


In [20]:

# 6. Inspeção de duplicatas e unicidade de order_id
# Leitura seletiva em lotes, preservando o Parquet original

from collections import Counter

frequencia_ids = Counter()
total_ids = 0
ids_nulos = 0

for lote in parquet.iter_batches(
    batch_size=10_000,
    columns=["order_id"]
):
    coluna_ids = lote.column(0)

    total_ids += lote.num_rows
    ids_nulos += coluna_ids.null_count

    # Acumula as frequências dos IDs não nulos
    frequencia_ids.update(
        valor
        for valor in coluna_ids.to_pylist()
        if valor is not None
    )

ids_distintos = len(frequencia_ids)

ids_repetidos = sum(
    1 for qtd in frequencia_ids.values()
    if qtd > 1
)

ocorrencias_duplicadas = sum(
    qtd - 1
    for qtd in frequencia_ids.values()
    if qtd > 1
)

print("=== UNICIDADE DE ORDER_ID ===")
print(f"Total de registros: {total_ids:,}")
print(f"IDs distintos: {ids_distintos:,}")
print(f"IDs nulos: {ids_nulos:,}")
print(f"IDs repetidos: {ids_repetidos:,}")
print(f"Ocorrências duplicadas: {ocorrencias_duplicadas:,}")

print("\n=== VALIDAÇÃO ===")
print(
    "Total consistente com metadados:",
    total_ids == metadata.num_rows
)
print(
    "Contagem consistente:",
    ids_distintos + ocorrencias_duplicadas
    + ids_nulos == total_ids
)


=== UNICIDADE DE ORDER_ID ===
Total de registros: 1,424,406
IDs distintos: 1,424,406
IDs nulos: 0
IDs repetidos: 0
Ocorrências duplicadas: 0

=== VALIDAÇÃO ===
Total consistente com metadados: True
Contagem consistente: True



### Passo 7 — Explicação do código e resultados

#### 1. Leitura do Parquet em lotes

`parquet.iter_batches(batch_size=10_000, columns=["order_id"])`

- `iter_batches()`: percorre o arquivo Parquet em lotes.
- `batch_size=10_000`: solicita lotes de até 10 mil registros.
- `columns=["order_id"]`: lê somente a coluna necessária.
- `lote.column(0)`: acessa a primeira coluna do lote.
- `lote.num_rows`: retorna a quantidade de linhas do lote.
- `coluna_ids.null_count`: conta os valores nulos.

**Por que usar lotes?**

Para evitar carregar todas as 19 colunas e os 1.424.406
registros simultaneamente na memória.

#### 2. Contagem de frequências com Counter

`from collections import Counter`

`Counter` é uma classe da biblioteca padrão do Python
utilizada para contar a frequência dos elementos.

Exemplo:

```python
from collections import Counter

ids = [101, 102, 101, 103, 102, 104]
frequencia = Counter(ids)

print(frequencia)
# Counter({101: 2, 102: 2, 103: 1, 104: 1})
```

No nosso código:

`frequencia_ids = Counter()`

Cria um contador inicialmente vazio.

`frequencia_ids.update(...)`

Atualiza as frequências dos identificadores a cada lote.

`coluna_ids.to_pylist()`

Converte os valores da coluna PyArrow para uma lista Python.

`if valor is not None`

Ignora os identificadores nulos na contagem de frequências.
Os nulos são contabilizados separadamente.

**Observação:** embora a leitura seja feita em lotes,
o Counter mantém os identificadores distintos na memória.

#### 3. Identificadores distintos

`ids_distintos = len(frequencia_ids)`

A função `len()` retorna a quantidade de chaves do Counter.

Cada chave representa um identificador diferente,
independentemente de quantas vezes ele aparece.

#### 4. Identificadores repetidos

```python
ids_repetidos = sum(
    1 for qtd in frequencia_ids.values()
    if qtd > 1
)
```

- `frequencia_ids.values()`: retorna as frequências.
- `for qtd in ...`: percorre cada frequência.
- `if qtd > 1`: seleciona IDs que aparecem mais de uma vez.
- `sum(1 for ...)`: soma 1 para cada ID repetido.

Essa construção é chamada de expressão geradora
(generator expression).

#### 5. Ocorrências duplicadas

```python
ocorrencias_duplicadas = sum(
    qtd - 1
    for qtd in frequencia_ids.values()
    if qtd > 1
)
```

Aqui, subtraímos 1 da frequência de cada identificador
repetido para desconsiderar sua primeira ocorrência.

Exemplo:

| ID | Frequência | Ocorrências duplicadas |
|---|---:|---:|
| 101 | 3 | 2 |
| 102 | 2 | 1 |
| 103 | 1 | 0 |

Nesse exemplo:

- IDs repetidos: 2 (101 e 102).
- Ocorrências duplicadas: 3.

#### 6. Validação dos resultados

`total_ids == metadata.num_rows`

Confirma se a quantidade processada corresponde ao
total informado nos metadados do Parquet.

`ids_distintos + ocorrencias_duplicadas + ids_nulos == total_ids`

Verifica se a soma dos identificadores distintos,
ocorrências duplicadas e nulos corresponde ao total
de registros.

O operador `==` realiza uma comparação e retorna
`True` ou `False`.

#### 7. Resultados obtidos

| Indicador | Resultado |
|---|---:|
| Total de registros | 1.424.406 |
| IDs distintos | 1.424.406 |
| IDs nulos | 0 |
| IDs repetidos | 0 |
| Ocorrências duplicadas | 0 |
| Consistência com metadados | True |
| Consistência das contagens | True |

#### 8. Conclusão técnica e de negócio

Todos os 1.424.406 registros possuem um `order_id`
único e não nulo.

**Conclusão técnica:**

- Não foram identificadas duplicidades de `order_id`.
- A coluna é candidata a chave primária no futuro ETL.
- A unicidade está validada para o arquivo Shanghai analisado.
- Não foi necessário alterar ou excluir registros.

**Interpretação de negócio:**

A unicidade permite contar registros de coleta por
`order_id` sem risco de contagem duplicada por esse campo.

Entretanto, ainda precisamos investigar os demais atributos
para confirmar a granularidade operacional dos registros
e interpretar corretamente os indicadores de negócio.

**Status: inspeção de unicidade concluída e validada.**



## Passo 8 — Identificadores e relacionamentos

### Objetivo

Investigar a cardinalidade dos principais identificadores
operacionais presentes no dataset LaDe-P Shanghai.

### Perguntas

- Quantos entregadores (`courier_id`) distintos existem?
- Quantas regiões (`region_id`) distintas existem?
- Quantas áreas operacionais (`aoi_id`) distintas existem?
- Existem valores nulos nesses identificadores?

### Metodologia

Utilizar PyArrow para ler somente as três colunas
necessárias, em lotes de 10.000 registros.

Acumular os valores distintos em conjuntos (`set`),
sem carregar as demais colunas do dataset.

Nesta primeira análise, não serão avaliados os
relacionamentos entre os identificadores.

Nenhum dado original será modificado.


In [21]:

# Passo 8 — Cardinalidade dos identificadores operacionais

# Um set armazena somente valores distintos.
identificadores = {
    "courier_id": set(),
    "region_id": set(),
    "aoi_id": set()
}

nulos = {coluna: 0 for coluna in identificadores}

# Lê somente as colunas necessárias, em lotes.
for lote in parquet.iter_batches(
    batch_size=10_000,
    columns=list(identificadores)
):
    for coluna in identificadores:
        valores = lote.column(coluna)

        # Contabiliza os valores ausentes.
        nulos[coluna] += valores.null_count

        # Adiciona ao set somente valores não nulos.
        identificadores[coluna].update(
            valor
            for valor in valores.to_pylist()
            if valor is not None
        )

print("=== CARDINALIDADE DOS IDENTIFICADORES ===")

for coluna, valores in identificadores.items():
    print(f"\nColuna: {coluna}")
    print(f"Valores distintos: {len(valores):,}")
    print(f"Valores nulos: {nulos[coluna]:,}")


=== CARDINALIDADE DOS IDENTIFICADORES ===

Coluna: courier_id
Valores distintos: 4,494
Valores nulos: 0

Coluna: region_id
Valores distintos: 30
Valores nulos: 0

Coluna: aoi_id
Valores distintos: 5,867
Valores nulos: 0



### Passo 8 — Explicação do código e resultados

#### 1. Estrutura `set` do Python

Um `set` é uma coleção que armazena somente valores
distintos, eliminando automaticamente repetições.

Exemplo:

```python
regioes = {10, 20, 10, 30, 20}
print(regioes)
# {10, 20, 30}
```

No código utilizamos:

```python
identificadores = {
    "courier_id": set(),
    "region_id": set(),
    "aoi_id": set()
}
```

Criamos um dicionário Python no qual cada chave
representa uma coluna e cada valor é um conjunto vazio.

**Diferença entre `set` e `Counter`:**

- `set`: identifica quais valores distintos existem.
- `Counter`: identifica os valores e quantas vezes aparecem.

#### 2. Compreensão de dicionário

```python
nulos = {coluna: 0 for coluna in identificadores}
```

Essa construção é chamada de *dict comprehension*.

Ela cria um dicionário com as mesmas chaves de
`identificadores`, inicializando cada contador com zero.

#### 3. Leitura seletiva com PyArrow

```python
parquet.iter_batches(
    batch_size=10_000,
    columns=list(identificadores)
)
```

- `iter_batches()`: percorre o Parquet em lotes.
- `batch_size`: define o tamanho máximo solicitado por lote.
- `list(identificadores)`: transforma as chaves do
  dicionário em uma lista de nomes de colunas.

Somente as três colunas necessárias são lidas.

#### 4. Percorrendo as colunas

```python
for coluna in identificadores:
    valores = lote.column(coluna)
```

O primeiro comando percorre os nomes das colunas.

O segundo recupera os valores da coluna correspondente
dentro do lote PyArrow.

#### 5. Contagem de nulos

```python
nulos[coluna] += valores.null_count
```

- `null_count`: quantidade de valores nulos no lote.
- `+=`: adiciona essa quantidade ao acumulador anterior.

#### 6. Acumulando valores distintos

```python
identificadores[coluna].update(
    valor
    for valor in valores.to_pylist()
    if valor is not None
)
```

- `to_pylist()`: converte a coluna em uma lista Python.
- `for valor in ...`: percorre os elementos.
- `if valor is not None`: ignora valores nulos.
- `update()`: adiciona os elementos ao conjunto.

Como utilizamos `set`, valores repetidos não são
armazenados novamente.

#### 7. Resultados obtidos

| Identificador | Distintos | Nulos |
|---|---:|---:|
| courier_id | 4.494 | 0 |
| region_id | 30 | 0 |
| aoi_id | 5.867 | 0 |

#### 8. Conclusão técnica

- Os três identificadores estão preenchidos.
- Foram identificados 4.494 entregadores distintos.
- Existem 30 regiões distintas.
- Existem 5.867 AOIs distintas.
- A análise utilizou leitura seletiva em lotes.
- Nenhum registro original foi modificado.

A ausência de nulos não garante que todos os
identificadores correspondam a entidades válidas.

#### 9. Possíveis aplicações de negócio

Esses identificadores poderão sustentar análises como:

- Volume de coletas por região.
- Distribuição de coletas por entregador.
- Concentração de coletas por AOI.
- Comparação de atrasos entre regiões.
- Identificação de áreas com maior demanda.

**Limitações atuais:**

Ainda não verificamos os relacionamentos entre
entregadores, regiões e AOIs.

Também não podemos concluir que cada AOI pertence
exclusivamente a uma região ou que cada entregador
atua em apenas uma região.

**Status: cardinalidade dos identificadores analisada.**



## Passo 9 — Relacionamentos entre identificadores

### Objetivo

Investigar os relacionamentos entre os identificadores
operacionais do dataset LaDe-P Shanghai.

### Primeira análise: entregadores e regiões

Perguntas:

- Um entregador realiza coletas em uma ou várias regiões?
- Quantos entregadores atuam em apenas uma região?
- Quantos atuam em múltiplas regiões?
- Qual o maior número de regiões associado a um entregador?

### Metodologia

Utilizar PyArrow para ler somente `courier_id` e
`region_id`, em lotes de 10.000 registros.

Construir um dicionário de conjuntos (`dict` de `set`),
associando cada entregador às regiões em que possui
registros de coleta.

Nenhum dado original será modificado.


In [22]:

# Passo 9 — Relacionamento entre courier_id e region_id

from collections import defaultdict, Counter

# Cada entregador terá um conjunto de regiões distintas.
regioes_por_entregador = defaultdict(set)

for lote in parquet.iter_batches(
    batch_size=10_000,
    columns=["courier_id", "region_id"]
):
    entregadores = lote.column("courier_id").to_pylist()
    regioes = lote.column("region_id").to_pylist()

    # Associa cada entregador à região de sua coleta.
    for entregador, regiao in zip(entregadores, regioes):
        if entregador is not None and regiao is not None:
            regioes_por_entregador[entregador].add(regiao)

# Quantas regiões distintas estão associadas a cada entregador?
quantidade_regioes = Counter(
    len(regioes)
    for regioes in regioes_por_entregador.values()
)

total_entregadores = len(regioes_por_entregador)

uma_regiao = quantidade_regioes[1]

multiplas_regioes = sum(
    qtd
    for numero_regioes, qtd in quantidade_regioes.items()
    if numero_regioes > 1
)

maximo_regioes = max(quantidade_regioes, default=0)

print("=== ENTREGADORES POR QUANTIDADE DE REGIÕES ===")

for numero_regioes, qtd in sorted(quantidade_regioes.items()):
    print(f"{numero_regioes} região(ões): {qtd:,} entregadores")

print("\n=== RESUMO ===")
print(f"Total de entregadores: {total_entregadores:,}")
print(f"Entregadores em uma região: {uma_regiao:,}")
print(f"Entregadores em múltiplas regiões: {multiplas_regioes:,}")
print(f"Máximo de regiões por entregador: {maximo_regioes}")

print("\n=== VALIDAÇÃO ===")
print(
    "Total consistente:",
    total_entregadores == len(identificadores["courier_id"])
)
print(
    "Distribuição consistente:",
    uma_regiao + multiplas_regioes == total_entregadores
)


=== ENTREGADORES POR QUANTIDADE DE REGIÕES ===
1 região(ões): 3,773 entregadores
2 região(ões): 610 entregadores
3 região(ões): 80 entregadores
4 região(ões): 28 entregadores
7 região(ões): 1 entregadores
12 região(ões): 1 entregadores
17 região(ões): 1 entregadores

=== RESUMO ===
Total de entregadores: 4,494
Entregadores em uma região: 3,773
Entregadores em múltiplas regiões: 721
Máximo de regiões por entregador: 17

=== VALIDAÇÃO ===
Total consistente: True
Distribuição consistente: True



### Passo 9 — Explicação do código e resultados

#### 1. Dicionário de conjuntos com `defaultdict`

```python
from collections import defaultdict, Counter

regioes_por_entregador = defaultdict(set)
```

`defaultdict` é uma classe da biblioteca padrão do Python
que cria automaticamente um valor inicial quando acessamos
uma chave ainda inexistente.

Com `defaultdict(set)`, cada novo entregador recebe
automaticamente um conjunto vazio.

Exemplo:

```python
regioes_por_entregador[101].add(10)
regioes_por_entregador[101].add(20)
regioes_por_entregador[101].add(10)
```

Resultado: o entregador 101 terá o conjunto `{10, 20}`.

O valor 10 não será duplicado porque utilizamos `set`.

#### 2. Leitura seletiva do Parquet

```python
for lote in parquet.iter_batches(
    batch_size=10_000,
    columns=["courier_id", "region_id"]
):
```

Lê somente as duas colunas necessárias, em lotes de
até 10.000 registros, evitando carregar todo o dataset.

#### 3. Associação entre entregadores e regiões

```python
for entregador, regiao in zip(entregadores, regioes):
    if entregador is not None and regiao is not None:
        regioes_por_entregador[entregador].add(regiao)
```

- `zip()`: combina elementos das duas listas por posição.
- `if ... is not None`: desconsidera valores nulos.
- `.add()`: adiciona a região ao conjunto do entregador.

Assim, construímos um relacionamento entre cada
`courier_id` e suas regiões distintas.

#### 4. Contagem de regiões por entregador

```python
quantidade_regioes = Counter(
    len(regioes)
    for regioes in regioes_por_entregador.values()
)
```

- `.values()`: retorna os conjuntos de regiões.
- `len(regioes)`: conta as regiões distintas do entregador.
- `Counter()`: contabiliza quantos entregadores possuem
  determinada quantidade de regiões.

Exemplo:

Se três entregadores possuem respectivamente 1, 1 e 2
regiões, o resultado será `Counter({1: 2, 2: 1})`.

#### 5. Contagem de entregadores com múltiplas regiões

```python
multiplas_regioes = sum(
    qtd
    for numero_regioes, qtd in quantidade_regioes.items()
    if numero_regioes > 1
)
```

Percorre os pares de quantidade de regiões e número
de entregadores, somando somente aqueles associados
a mais de uma região.

#### 6. Resultados obtidos

| Regiões distintas | Entregadores |
|---|---:|
| 1 | 3.773 |
| 2 | 610 |
| 3 | 80 |
| 4 | 28 |
| 7 | 1 |
| 12 | 1 |
| 17 | 1 |
| **Total** | **4.494** |

- Uma região: 3.773 entregadores (83,96%).
- Múltiplas regiões: 721 entregadores (16,04%).
- Máximo observado: 17 regiões por entregador.
- Validação dos totais: True.
- Validação da distribuição: True.

#### 7. Conclusão técnica

O relacionamento entre `courier_id` e `region_id`
não é exclusivamente de um entregador para uma região.

Existem entregadores associados a múltiplas regiões
ao longo dos registros analisados.

Essa característica deverá ser considerada na
modelagem dos futuros indicadores e dashboards.

#### 8. Interpretação de negócio

A maioria dos entregadores aparece em uma única região.

Entretanto, três entregadores possuem registros em
7, 12 e 17 regiões distintas.

Esses casos merecem investigação adicional para
compreender se representam mudanças de área ao longo
do tempo, características operacionais ou inconsistências.

Não podemos concluir que esses entregadores circularam
fisicamente entre todas essas regiões em um mesmo dia.

#### 9. Perguntas para análises futuras

- Em quais datas esses entregadores atuaram?
- Quantas regiões diferentes aparecem no mesmo dia?
- Quantas coletas foram realizadas em cada região?
- Existem diferenças nos atrasos por região?
- As associações entre entregadores e regiões mudam
  ao longo do período analisado?

**Status: relacionamento courier_id × region_id analisado.**



## Passo 10 — Relacionamento entre AOIs e regiões

### Objetivo

Investigar a cardinalidade do relacionamento entre
`aoi_id` e `region_id` no dataset LaDe-P Shanghai.

### Perguntas

- Cada AOI está associada a uma única região?
- Existem AOIs associadas a múltiplas regiões?
- Quantas AOIs possuem essa característica?
- Qual o maior número de regiões associado a uma AOI?

### Metodologia

Utilizar PyArrow para ler somente as colunas
`aoi_id` e `region_id`, em lotes de 10.000 registros.

Construir um dicionário de conjuntos (`defaultdict(set)`)
para armazenar as regiões distintas associadas a cada AOI.

Não serão realizadas alterações nos dados originais.

### Aplicação futura

Verificar se `aoi_id` pode ser utilizado como uma
subdivisão de `region_id` nos dashboards e mapas.


In [23]:

# Passo 10 — Relacionamento entre aoi_id e region_id

from collections import defaultdict, Counter

# Cada AOI terá um conjunto de regiões distintas.
regioes_por_aoi = defaultdict(set)

for lote in parquet.iter_batches(
    batch_size=10_000,
    columns=["aoi_id", "region_id"]
):
    aois = lote.column("aoi_id").to_pylist()
    regioes = lote.column("region_id").to_pylist()

    # Associa cada AOI às regiões encontradas nos registros.
    for aoi, regiao in zip(aois, regioes):
        if aoi is not None and regiao is not None:
            regioes_por_aoi[aoi].add(regiao)

# Conta quantas regiões distintas cada AOI possui.
quantidade_regioes_por_aoi = Counter(
    len(regioes)
    for regioes in regioes_por_aoi.values()
)

total_aois = len(regioes_por_aoi)

aois_uma_regiao = quantidade_regioes_por_aoi[1]

aois_multiplas_regioes = sum(
    qtd
    for numero_regioes, qtd in quantidade_regioes_por_aoi.items()
    if numero_regioes > 1
)

maximo_regioes_por_aoi = max(
    quantidade_regioes_por_aoi,
    default=0
)

print("=== AOIs POR QUANTIDADE DE REGIÕES ===")

for numero_regioes, qtd in sorted(
    quantidade_regioes_por_aoi.items()
):
    print(f"{numero_regioes} região(ões): {qtd:,} AOIs")

print("\n=== RESUMO ===")
print(f"Total de AOIs: {total_aois:,}")
print(f"AOIs em uma região: {aois_uma_regiao:,}")
print(f"AOIs em múltiplas regiões: {aois_multiplas_regioes:,}")
print(f"Máximo de regiões por AOI: {maximo_regioes_por_aoi}")

print("\n=== VALIDAÇÃO ===")
print(
    "Total consistente:",
    total_aois == len(identificadores["aoi_id"])
)
print(
    "Distribuição consistente:",
    aois_uma_regiao + aois_multiplas_regioes == total_aois
)


=== AOIs POR QUANTIDADE DE REGIÕES ===
1 região(ões): 5,768 AOIs
2 região(ões): 94 AOIs
3 região(ões): 4 AOIs
4 região(ões): 1 AOIs

=== RESUMO ===
Total de AOIs: 5,867
AOIs em uma região: 5,768
AOIs em múltiplas regiões: 99
Máximo de regiões por AOI: 4

=== VALIDAÇÃO ===
Total consistente: True
Distribuição consistente: True



### Passo 10 — Explicação do código e resultados

#### 1. Dicionário de conjuntos com `defaultdict`

```python
regioes_por_aoi = defaultdict(set)
```

Cria um dicionário em que cada chave será um `aoi_id`
e cada valor será um conjunto (`set`) de regiões.

O `defaultdict(set)` cria automaticamente um conjunto
vazio quando uma AOI aparece pela primeira vez.

Exemplo:

```python
regioes_por_aoi[501].add(10)
regioes_por_aoi[501].add(20)
regioes_por_aoi[501].add(10)
```

Resultado: a AOI 501 estará associada ao conjunto
`{10, 20}`, sem repetir a região 10.

#### 2. Leitura seletiva do Parquet

```python
for lote in parquet.iter_batches(
    batch_size=10_000,
    columns=["aoi_id", "region_id"]
):
```

- `iter_batches()`: percorre o arquivo em lotes.
- `batch_size=10_000`: solicita até 10 mil registros por lote.
- `columns`: seleciona somente as duas colunas necessárias.

Isso reduz o volume de dados carregados simultaneamente.

#### 3. Associação entre AOIs e regiões

```python
for aoi, regiao in zip(aois, regioes):
    if aoi is not None and regiao is not None:
        regioes_por_aoi[aoi].add(regiao)
```

- `zip()`: combina os elementos das duas listas por posição.
- `is not None`: desconsidera identificadores nulos.
- `.add()`: adiciona a região ao conjunto da AOI.

Assim, cada AOI passa a ter seu conjunto de regiões
distintas observadas no dataset.

#### 4. Distribuição da quantidade de regiões

```python
quantidade_regioes_por_aoi = Counter(
    len(regioes)
    for regioes in regioes_por_aoi.values()
)
```

- `.values()`: retorna os conjuntos de regiões.
- `len(regioes)`: conta quantas regiões cada AOI possui.
- `Counter()`: contabiliza quantas AOIs apresentam
  determinada quantidade de regiões.

#### 5. Contagem de AOIs em múltiplas regiões

```python
aois_multiplas_regioes = sum(
    qtd
    for numero_regioes, qtd in quantidade_regioes_por_aoi.items()
    if numero_regioes > 1
)
```

A expressão percorre os pares do Counter e soma
somente as AOIs associadas a mais de uma região.

#### 6. Identificação do máximo

```python
maximo_regioes_por_aoi = max(
    quantidade_regioes_por_aoi,
    default=0
)
```

`max()` retorna a maior chave do Counter, ou seja,
a maior quantidade de regiões observada para uma AOI.

`default=0` evita erro caso o Counter esteja vazio.

#### 7. Resultados obtidos

| Regiões distintas | Quantidade de AOIs |
|---|---:|
| 1 | 5.768 |
| 2 | 94 |
| 3 | 4 |
| 4 | 1 |
| **Total** | **5.867** |

- AOIs em uma região: 5.768 (98,31%).
- AOIs em múltiplas regiões: 99 (1,69%).
- Máximo observado: 4 regiões por AOI.
- Validação dos totais: True.
- Validação da distribuição: True.

#### 8. Conclusão técnica

O relacionamento entre `aoi_id` e `region_id`
não é exclusivamente de muitos para um.

Embora a maioria das AOIs apareça em apenas uma região,
99 AOIs estão associadas a múltiplas regiões.

Portanto, não devemos assumir que `aoi_id` determina
unicamente `region_id` no conjunto analisado.

Isso deverá ser considerado na futura modelagem
dimensional e na construção dos dashboards.

#### 9. Interpretação de negócio

A maioria das AOIs possui associação com uma única
região nos registros analisados.

Entretanto, algumas aparecem em duas, três ou quatro
regiões distintas.

Ainda não sabemos se isso representa:

- Mudanças de associação ao longo do tempo.
- Características da organização operacional.
- Identificadores reutilizados em diferentes regiões.
- Possíveis inconsistências nos dados.

Não é possível determinar a causa apenas com
a contagem de regiões distintas por AOI.

#### 10. Perguntas para investigações futuras

- Quais são as 99 AOIs associadas a múltiplas regiões?
- Essas associações ocorrem nas mesmas datas?
- Quantas coletas existem em cada combinação AOI/região?
- Há diferenças nos atrasos entre essas combinações?
- A identificação de uma AOI exige uma chave composta
  por `region_id` e `aoi_id`?

**Status: relacionamento aoi_id × region_id analisado.**



## Passo 11 — AOIs associadas a múltiplas regiões por dia

### Objetivo

Investigar se uma mesma AOI (`aoi_id`) aparece associada
a diferentes regiões (`region_id`) em um mesmo dia.

### Perguntas

- Existem AOIs associadas a múltiplas regiões no mesmo dia?
- Quantas AOIs apresentam esse comportamento?
- Quantas combinações AOI/data possuem múltiplas regiões?

### Metodologia

Utilizar PyArrow para ler somente `aoi_id`, `region_id`
e `ds`, em lotes de 10.000 registros.

Construir um dicionário de conjuntos para associar
cada combinação (AOI, data) às regiões distintas.

A análise considera a data registrada em `ds`.

Nenhum dado original será modificado.


In [24]:

# Passo 11 — AOIs em múltiplas regiões no mesmo dia

from collections import defaultdict

# Chave: (aoi_id, ds)
# Valor: conjunto de regiões distintas
regioes_por_aoi_dia = defaultdict(set)

for lote in parquet.iter_batches(
    batch_size=10_000,
    columns=["aoi_id", "region_id", "ds"]
):
    aois = lote.column("aoi_id").to_pylist()
    regioes = lote.column("region_id").to_pylist()
    datas = lote.column("ds").to_pylist()

    # Associa cada AOI/data às regiões observadas.
    for aoi, regiao, data in zip(aois, regioes, datas):
        if aoi is not None and regiao is not None and data is not None:
            regioes_por_aoi_dia[(aoi, data)].add(regiao)

# Combinações AOI/data associadas a mais de uma região
combinacoes_multiplas = {
    chave: regioes
    for chave, regioes in regioes_por_aoi_dia.items()
    if len(regioes) > 1
}

# AOIs distintas que apresentam múltiplas regiões no mesmo dia
aois_multiplas_mesmo_dia = {
    aoi
    for aoi, data in combinacoes_multiplas
}

print("=== AOIs EM MÚLTIPLAS REGIÕES NO MESMO DIA ===")
print(f"Total de combinações AOI/data: {len(regioes_por_aoi_dia):,}")
print(f"Combinações com múltiplas regiões: {len(combinacoes_multiplas):,}")
print(f"AOIs distintas nessa condição: {len(aois_multiplas_mesmo_dia):,}")

print("\n=== VALIDAÇÃO ===")
print(
    "AOIs dentro do conjunto anteriormente identificado:",
    aois_multiplas_mesmo_dia.issubset(
        {
            aoi
            for aoi, regioes in regioes_por_aoi.items()
            if len(regioes) > 1
        }
    )
)


=== AOIs EM MÚLTIPLAS REGIÕES NO MESMO DIA ===
Total de combinações AOI/data: 342,300
Combinações com múltiplas regiões: 148
AOIs distintas nessa condição: 66

=== VALIDAÇÃO ===
AOIs dentro do conjunto anteriormente identificado: True



### Passo 11 — Explicação do código e interpretação dos resultados

#### 1. Conceitos Python utilizados

**`defaultdict(set)`**

Cria um dicionário no qual cada nova chave recebe automaticamente
um conjunto vazio (`set`).

Neste caso:
- Chave: `(aoi_id, ds)`
- Valor: conjunto de regiões associadas à AOI naquela data.

O conjunto elimina automaticamente regiões repetidas.

**Tupla `(aoi, data)`**

Permite utilizar dois campos como uma única chave do dicionário.
Assim, conseguimos analisar cada AOI separadamente por data.

**`zip(aois, regioes, datas)`**

Percorre simultaneamente as três listas, associando os valores
correspondentes de cada registro.

**Compreensão de dicionário**

Seleciona somente as combinações AOI/data cujo conjunto
contém mais de uma região:

`if len(regioes) > 1`

**Compreensão de conjunto**

Extrai as AOIs distintas das combinações identificadas,
evitando contar uma mesma AOI várias vezes.

**`issubset()`**

Verifica se todas as AOIs identificadas nesta análise
pertencem ao conjunto de AOIs anteriormente classificadas
como associadas a múltiplas regiões.

#### 2. Resultados obtidos

| Indicador | Resultado |
|---|---:|
| Combinações distintas AOI/data | 342.300 |
| Combinações com múltiplas regiões | 148 |
| AOIs distintas nessa condição | 66 |
| AOIs com múltiplas regiões na análise anterior | 99 |
| Validação de consistência | True |

Das 99 AOIs anteriormente associadas a múltiplas regiões,
66 apresentam esse comportamento em pelo menos uma mesma data.

As outras 33 AOIs não apresentaram múltiplas regiões
dentro de uma mesma data nos registros analisados.

As 148 combinações representam aproximadamente 0,043%
das 342.300 combinações AOI/data.

#### 3. Interpretação técnica

O resultado demonstra que a associação entre AOI e região
não é necessariamente única, mesmo considerando a data `ds`.

Portanto, não devemos assumir que cada AOI pertence a uma
única região por dia.

A validação retornou `True`, confirmando a consistência
entre os conjuntos identificados nos Passos 10 e 11.

#### 4. Interpretação de negócio

**Fato observado:**
66 AOIs aparecem associadas a mais de uma região
em pelo menos uma mesma data registrada em `ds`.

**Hipóteses possíveis:**
- As regiões podem representar classificações operacionais
  que não são exclusivas.
- Uma AOI pode ser atendida por diferentes regiões.
- O campo `ds` pode representar uma data de referência
  diferente do momento operacional da coleta.

**Informações adicionais necessárias:**
- Definição oficial de `region_id` e `aoi_id`.
- Significado operacional do campo `ds`.
- Regras utilizadas para associar AOIs às regiões.

Não é possível afirmar que essas associações sejam erros
de qualidade dos dados.

#### 5. Aplicação futura no Power BI

Esses resultados poderão ser apresentados em uma página
exploratória com indicadores, tabelas e filtros por região,
AOI e período.

O dashboard permitirá investigar as associações e comparar
os resultados visuais com as verificações realizadas em Python.



## Passo 12 — Validação semântica dos campos temporais

### 12.1 — Inspeção de `ds` e `pickup_time`

#### Objetivo

Investigar como os campos `ds` e `pickup_time` estão efetivamente armazenados no dataset LaDe-P Shanghai, antes de utilizá-los em análises de períodos, SLA e atrasos.

A documentação do dataset associa `ds` à data da coleta. Entretanto, precisamos confirmar como essa informação aparece nos registros reais.

Nesta primeira inspeção, verificaremos:

- O tipo físico de cada coluna no schema do PyArrow.
- Os primeiros valores não nulos de `ds` e `pickup_time`.
- A representação Python dos valores encontrados.
- Se os valores aparentam conter data completa, apenas horário ou outra codificação.

#### Conceitos Python utilizados

**`parquet.schema_arrow`**

Retorna o schema do arquivo Parquet interpretado pelo PyArrow, permitindo consultar os tipos de dados das colunas sem carregar todos os registros.

**`iter_batches()`**

Realiza a leitura do Parquet em lotes, permitindo selecionar apenas as colunas necessárias.

**`to_pylist()`**

Converte os valores de uma coluna PyArrow em uma lista Python, facilitando sua inspeção.

**`type()`**

Identifica o tipo Python de um valor, como `str`, `int`, `datetime` ou `date`.

**`repr()`**

Mostra a representação do valor, ajudando a identificar diferenças entre strings, números e objetos de data/hora.

#### Metodologia

Serão lidas apenas as colunas `ds` e `pickup_time`, com um limite de dez exemplos não nulos por coluna.

Esta inspeção é exploratória: os exemplos não representam necessariamente todos os formatos presentes no dataset.

Nenhum dado será modificado, convertido ou excluído.

#### Resultado esperado

Identificar a estrutura inicial dos dois campos para orientar a próxima validação: verificar se `ds` corresponde à data registrada em `pickup_time`.


In [25]:

# Passo 12.1 — Inspeção inicial de ds e pickup_time
# Leitura seletiva, sem modificar o Parquet original

colunas_temporais = ["ds", "pickup_time"]
limite_exemplos = 10

print("=== TIPOS NO SCHEMA PYARROW ===")

for nome in colunas_temporais:
    campo = parquet.schema_arrow.field(nome)
    print(f"{nome}: {campo.type}")

print("\n=== EXEMPLOS DE VALORES REAIS ===")

exemplos = {nome: [] for nome in colunas_temporais}

for lote in parquet.iter_batches(
    batch_size=1_000,
    columns=colunas_temporais
):
    for nome in colunas_temporais:
        valores = lote.column(nome).to_pylist()

        for valor in valores:
            if valor is not None:
                if len(exemplos[nome]) < limite_exemplos:
                    exemplos[nome].append(valor)

    if all(
        len(valores) >= limite_exemplos
        for valores in exemplos.values()
    ):
        break

for nome, valores in exemplos.items():
    print(f"\nColuna: {nome}")

    for indice, valor in enumerate(valores, start=1):
        print(
            f"{indice:02d}. "
            f"valor={repr(valor)} | "
            f"tipo_python={type(valor).__name__}"
        )


=== TIPOS NO SCHEMA PYARROW ===
ds: int64
pickup_time: string

=== EXEMPLOS DE VALORES REAIS ===

Coluna: ds
01. valor=708 | tipo_python=int
02. valor=721 | tipo_python=int
03. valor=712 | tipo_python=int
04. valor=709 | tipo_python=int
05. valor=707 | tipo_python=int
06. valor=719 | tipo_python=int
07. valor=712 | tipo_python=int
08. valor=728 | tipo_python=int
09. valor=717 | tipo_python=int
10. valor=725 | tipo_python=int

Coluna: pickup_time
01. valor='07-08 10:10:00' | tipo_python=str
02. valor='07-21 10:10:00' | tipo_python=str
03. valor='07-12 17:22:00' | tipo_python=str
04. valor='07-09 15:54:00' | tipo_python=str
05. valor='07-07 09:53:00' | tipo_python=str
06. valor='07-19 09:47:00' | tipo_python=str
07. valor='07-12 09:35:00' | tipo_python=str
08. valor='07-28 14:53:00' | tipo_python=str
09. valor='07-17 09:29:00' | tipo_python=str
10. valor='07-25 11:02:00' | tipo_python=str



### 12.2 — Validação da correspondência entre `ds` e `pickup_time`

#### Objetivo

Verificar se o campo `ds`, armazenado como `int64`, corresponde ao mês e ao dia registrados em `pickup_time`, armazenado como `string`.

Na inspeção anterior, observamos:

- `ds = 708`
- `pickup_time = "07-08 10:10:00"`

Esses valores sugerem que `ds` utiliza a codificação numérica `MMDD`, enquanto `pickup_time` contém mês, dia e horário, sem o ano.

Essa interpretação ainda precisa ser validada em todos os registros.

#### Regra de comparação

A partir de `pickup_time`, extrair:

- Mês: posições 0 e 1.
- Dia: posições 3 e 4.

Calcular:

`ds_calculado = mes * 100 + dia`

Exemplo:

`07-08 10:10:00 → 7 * 100 + 8 = 708`

Comparar o resultado com o valor original de `ds`.

#### Conceitos Python utilizados

**`iter_batches()`**

Permite percorrer o Parquet em lotes, lendo somente as colunas necessárias.

**`pyarrow.compute`**

Disponibiliza operações vetorizadas para manipular e comparar valores sem converter todos os registros em objetos Python.

**`pc.utf8_slice_codeunits()`**

Extrai partes de uma string utilizando posições definidas.

**`pc.cast()`**

Converte os valores extraídos para um tipo numérico, permitindo calcular a codificação `MMDD`.

**`pc.equal()`**

Compara os valores de duas colunas, identificando correspondências.

**`pc.sum()`**

Conta os resultados verdadeiros de uma comparação booleana.

#### Metodologia

1. Ler somente `ds` e `pickup_time`.
2. Verificar valores nulos e formatos inválidos.
3. Extrair mês e dia de `pickup_time`.
4. Calcular a representação numérica `MMDD`.
5. Comparar com o valor original de `ds`.
6. Contabilizar correspondências e divergências.
7. Validar se a soma das categorias corresponde ao total de registros.

A validação será realizada sobre todo o dataset, utilizando lotes de 10.000 registros.

#### Premissas e limitações

- Nenhum ano será atribuído aos registros.
- Nenhuma transformação será gravada no arquivo original.
- Valores inválidos serão contabilizados separadamente.
- A correspondência entre os campos não comprova, isoladamente, o significado operacional de `ds`.
- A validação considera o padrão textual `MM-DD HH:MM:SS`.

#### Resultado esperado

Determinar se `ds` corresponde sistematicamente ao mês e ao dia de `pickup_time` e identificar eventuais divergências para investigação posterior.


In [26]:

# Passo 12.2 — Validação de ds × pickup_time
# Leitura em lotes, sem modificar o Parquet original

import pyarrow as pa
import pyarrow.compute as pc

total_registros = 0
correspondentes = 0
divergentes = 0
invalidos = 0

exemplos_divergentes = []
limite_exemplos = 10

for lote in parquet.iter_batches(
    batch_size=10_000,
    columns=["ds", "pickup_time"]
):
    ds = lote.column("ds")
    pickup = lote.column("pickup_time")

    total_registros += lote.num_rows

    # Validar o formato MM-DD HH:MM:SS.
    formato_valido = pc.match_substring_regex(
        pickup,
        r"^(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01]) "
        r"([01][0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$"
    )

    # Excluir registros nulos ou com formato inválido.
    valido = pc.and_(
        pc.is_valid(ds),
        pc.fill_null(formato_valido, False)
    )

    qtd_validos = pc.sum(valido).as_py() or 0
    invalidos += lote.num_rows - qtd_validos

    # Processar somente registros válidos.
    ds_validos = pc.filter(ds, valido)
    pickup_validos = pc.filter(pickup, valido)

    if len(ds_validos) == 0:
        continue

    # Extrair mês e dia do texto.
    mes_texto = pc.utf8_slice_codeunits(
        pickup_validos, start=0, stop=2
    )

    dia_texto = pc.utf8_slice_codeunits(
        pickup_validos, start=3, stop=5
    )

    mes = pc.cast(mes_texto, pa.int64())
    dia = pc.cast(dia_texto, pa.int64())

    # Calcular MMDD.
    ds_calculado = pc.add(
        pc.multiply(mes, 100),
        dia
    )

    # Comparar com ds original.
    iguais = pc.equal(ds_validos, ds_calculado)

    qtd_iguais = pc.sum(iguais).as_py() or 0

    correspondentes += qtd_iguais
    divergentes += len(ds_validos) - qtd_iguais

    # Guardar até 10 exemplos de divergência.
    if len(exemplos_divergentes) < limite_exemplos:
        diferentes = pc.invert(iguais)

        ds_div = pc.filter(ds_validos, diferentes).to_pylist()
        pickup_div = pc.filter(
            pickup_validos, diferentes
        ).to_pylist()
        calculado_div = pc.filter(
            ds_calculado, diferentes
        ).to_pylist()

        espacos = limite_exemplos - len(exemplos_divergentes)

        exemplos_divergentes.extend(
            list(zip(ds_div, pickup_div, calculado_div))[:espacos]
        )

print("=== VALIDAÇÃO DS × PICKUP_TIME ===")
print(f"Total de registros: {total_registros:,}")
print(f"Correspondentes: {correspondentes:,}")
print(f"Divergentes: {divergentes:,}")
print(f"Nulos ou formato inválido: {invalidos:,}")

print("\n=== PERCENTUAIS ===")

if total_registros > 0:
    print(
        f"Correspondentes: "
        f"{correspondentes / total_registros * 100:.2f}%"
    )
    print(
        f"Divergentes: "
        f"{divergentes / total_registros * 100:.2f}%"
    )
    print(
        f"Inválidos: "
        f"{invalidos / total_registros * 100:.2f}%"
    )

print("\n=== EXEMPLOS DE DIVERGÊNCIA ===")

if exemplos_divergentes:
    for ds_original, pickup_original, ds_esperado in exemplos_divergentes:
        print(
            f"ds={ds_original} | "
            f"pickup_time={pickup_original} | "
            f"MMDD_calculado={ds_esperado}"
        )
else:
    print("Nenhuma divergência encontrada.")

print("\n=== VALIDAÇÕES ===")

print(
    "Total consistente com metadados:",
    total_registros == metadata.num_rows
)

print(
    "Categorias consistentes:",
    correspondentes + divergentes + invalidos == total_registros
)


=== VALIDAÇÃO DS × PICKUP_TIME ===
Total de registros: 1,424,406
Correspondentes: 1,424,406
Divergentes: 0
Nulos ou formato inválido: 0

=== PERCENTUAIS ===
Correspondentes: 100.00%
Divergentes: 0.00%
Inválidos: 0.00%

=== EXEMPLOS DE DIVERGÊNCIA ===
Nenhuma divergência encontrada.

=== VALIDAÇÕES ===
Total consistente com metadados: True
Categorias consistentes: True



### 12.3 — Documentação oficial e premissas temporais

#### Objetivo

Confrontar as características observadas no arquivo LaDe-P Shanghai com as informações publicadas pelos autores do dataset.

Esta etapa busca estabelecer uma referência documental para a interpretação dos campos temporais antes de realizar conversões, cálculos de duração ou indicadores de SLA.

#### Fontes oficiais

- Dataset: https://huggingface.co/datasets/Cainiao-AI/LaDe-P
- Repositório: https://github.com/wenhaomin/LaDe
- Site do projeto: https://cainiaotechai.github.io/LaDe-website/

#### Período de referência

O projeto LaDe descreve dados operacionais coletados entre maio e outubro de 2022.

Período geral informado:

- Início: 01/05/2022
- Fim: 31/10/2022

Esse intervalo é uma referência documental do projeto. Ainda precisamos verificar a distribuição temporal específica do arquivo Shanghai.

#### Resultados já observados

| Campo | Tipo PyArrow | Formato observado |
|---|---|---|
| ds | int64 | MMDD, como 708 |
| pickup_time | string | MM-DD HH:MM:SS |

A validação quantitativa demonstrou que `ds` corresponde ao mês e dia de `pickup_time` em todos os 1.424.406 registros analisados.

- Correspondentes: 1.424.406
- Divergentes: 0
- Nulos ou formato inválido: 0
- Correspondência: 100%

#### Interpretação temporal

A documentação fornece 2022 como ano de referência do projeto.

Entretanto, o ano não está explicitamente armazenado nos valores de `pickup_time` examinados.

A utilização de 2022 nas futuras conversões deverá ser registrada como uma premissa fundamentada na documentação e validada contra o período observado no arquivo Shanghai.

#### Pontos pendentes

1. Verificar os meses e dias existentes no arquivo Shanghai.
2. Verificar se todos os registros respeitam o período documentado.
3. Investigar a consistência entre `accept_time` e `pickup_time`.
4. Verificar a validade das datas e possíveis transições de dia.
5. Investigar o fuso horário e as regras necessárias para calcular durações e SLA.

#### Conclusão parcial

A codificação `MMDD` de `ds` está validada empiricamente.

O ano de 2022 possui respaldo na documentação do projeto, mas sua aplicação aos registros específicos de Shanghai ainda requer validação.

Nenhuma transformação dos dados originais foi realizada.

**Status: documentação temporal registrada; validação do período do arquivo pendente.**



### 12.4.1 — Inventário dos campos temporais

#### Objetivo

Identificar os tipos de armazenamento e os formatos observados
nos campos temporais do LaDe-P Shanghai.

#### Campos investigados

- `accept_time`: horário de aceite da solicitação.
- `pickup_time`: horário da coleta.
- `time_window_start`: início da janela de coleta.
- `time_window_end`: fim da janela de coleta.
- `accept_gps_time`: horário do registro GPS associado ao aceite.
- `pickup_gps_time`: horário do registro GPS associado à coleta.
- `ds`: referência numérica de mês e dia da coleta.

#### Metodologia

1. Consultar os tipos definidos no schema PyArrow.
2. Examinar uma amostra de valores de cada campo.
3. Identificar os formatos aparentes.
4. Registrar diferenças entre os formatos observados.

Esta etapa é exclusivamente exploratória. Não serão realizadas
conversões, correções ou alterações nos dados originais.


In [27]:

# Passo 12.4.1 — Inventário dos campos temporais

campos_temporais = [
    "accept_time",
    "pickup_time",
    "time_window_start",
    "time_window_end",
    "accept_gps_time",
    "pickup_gps_time",
    "ds",
]

print("=== INVENTÁRIO DOS CAMPOS TEMPORAIS ===")

# Leitura de apenas um lote para inspecionar exemplos.
lote = next(
    parquet.iter_batches(
        batch_size=1000,
        columns=campos_temporais
    )
)

for campo in campos_temporais:
    tipo = schema.field(campo).type
    valores = lote.column(campo).to_pylist()

    exemplos = list(dict.fromkeys(
        valor for valor in valores if valor is not None
    ))[:5]

    print(f"\nCampo: {campo}")
    print(f"Tipo PyArrow: {tipo}")
    print(f"Exemplos: {exemplos}")


=== INVENTÁRIO DOS CAMPOS TEMPORAIS ===

Campo: accept_time
Tipo PyArrow: string
Exemplos: ['07-08 08:13:00', '07-21 08:14:00', '07-12 07:40:00', '07-09 15:38:00', '07-07 07:25:00']

Campo: pickup_time
Tipo PyArrow: string
Exemplos: ['07-08 10:10:00', '07-21 10:10:00', '07-12 17:22:00', '07-09 15:54:00', '07-07 09:53:00']

Campo: time_window_start
Tipo PyArrow: string
Exemplos: ['07-08 09:00:00', '07-21 09:00:00', '07-12 17:00:00', '07-09 17:00:00', '07-07 09:00:00']

Campo: time_window_end
Tipo PyArrow: string
Exemplos: ['07-08 11:00:00', '07-21 11:00:00', '07-12 19:00:00', '07-09 19:00:00', '07-07 11:00:00']

Campo: accept_gps_time
Tipo PyArrow: string
Exemplos: ['07-21 08:14:00', '07-12 07:37:00', '07-09 15:38:00', '07-07 07:25:00', '07-19 08:06:00']

Campo: pickup_gps_time
Tipo PyArrow: string
Exemplos: ['07-08 10:10:00', '07-21 10:10:00', '07-12 17:22:00', '07-09 15:54:00', '07-07 09:53:00']

Campo: ds
Tipo PyArrow: int64
Exemplos: [708, 721, 712, 709, 707]



### 12.4.1.1 — Resultados e conclusões

O inventário identificou sete campos relacionados a datas e horários.

**Resultados:**

- Seis campos são armazenados como `string`.
- Os exemplos desses seis campos seguem `MM-DD HH:MM:SS`.
- O campo `ds` é armazenado como `int64`, com codificação `MMDD`.
- Nenhum dos campos examinados apresenta o ano explicitamente.
- A documentação do LaDe fornece 2022 como referência temporal.

**Limitações:**

- A inspeção utilizou uma amostra de 1.000 registros.
- A padronização dos formatos ainda não foi validada em todo o dataset.
- A validade das datas e horários ainda não foi verificada.
- A consistência cronológica entre eventos ainda não foi avaliada.

**Conclusão:**

Os campos temporais apresentam formatos aparentemente padronizados.
A próxima investigação deverá verificar a validade desses formatos
em todos os registros, distinguindo valores ausentes de valores inválidos.

**Status: inventário inicial concluído.**



## 12.4 — Análise completa dos campos temporais

### Objetivo

Realizar uma investigação quantitativa, semântica e estatística dos
campos temporais do dataset LaDe-P Shanghai (Pickup).

A análise tem caráter exclusivamente exploratório (AED).
Nenhum registro será corrigido, excluído ou modificado.

### 1. Campos analisados

| Campo | Significado |
|---|---|
| `accept_time` | Data e horário do aceite da solicitação |
| `pickup_time` | Data e horário da coleta realizada |
| `time_window_start` | Início da janela prevista para coleta |
| `time_window_end` | Fim da janela prevista para coleta |
| `accept_gps_time` | Horário do registro GPS associado ao aceite |
| `pickup_gps_time` | Horário do registro GPS associado à coleta |
| `ds` | Referência numérica de mês e dia da coleta |

### 2. Referência documental

A documentação do projeto LaDe informa dados coletados entre
maio e outubro de 2022.

Fontes:

- https://github.com/wenhaomin/LaDe
- https://huggingface.co/datasets/Cainiao-AI/LaDe-P

Os seis campos temporais são armazenados como strings no formato
aparente `MM-DD HH:MM:SS`, sem ano explícito.

O campo `ds` é armazenado como inteiro no formato `MMDD`.

Para permitir cálculos de duração, utilizaremos **2022 como ano
de referência documental**, sem modificar os dados originais.

Essa premissa será registrada e avaliada nos resultados.

### 3. Verificações de qualidade

Para cada campo temporal, serão contabilizados:

- Total de registros.
- Valores preenchidos e nulos.
- Valores com formato válido e inválido.
- Datas impossíveis, como 02-30.
- Valores fora do período documental.
- Datas mínima e máxima observadas.

A verificação distingue formato textual válido de data
efetivamente válida no calendário.

### 4. Verificações de consistência

Serão avaliadas as seguintes relações:

**Aceite e coleta**

`accept_time <= pickup_time`

Uma coleta anterior ao aceite poderá indicar uma inconsistência
temporal que exigirá investigação.

**Janela de coleta**

`time_window_start <= time_window_end`

**Classificação da coleta**

- Antecipada: coleta antes do início da janela.
- Dentro da janela: coleta entre início e fim, inclusive.
- Atrasada: coleta após o fim da janela.

Essas categorias descrevem o comportamento observado.
A classificação definitiva de SLA depende das regras de negócio.

**Consistência de ds**

`ds = mês(pickup_time) * 100 + dia(pickup_time)`

Essa relação já foi validada anteriormente em 100% dos registros.
A presente análise reutilizará essa evidência, sem repetir a contagem.

### 5. Registros GPS

Serão analisados:

- Disponibilidade dos horários GPS.
- Diferença entre `accept_time` e `accept_gps_time`.
- Diferença entre `pickup_time` e `pickup_gps_time`.
- Diferenças negativas, positivas e iguais a zero.

Os horários GPS representam eventos de localização associados
aos eventos operacionais e não necessariamente seus instantes exatos.

### 6. Análise estatística

Para as durações calculáveis, serão apresentados:

- Quantidade de observações válidas.
- Média e mediana.
- Desvio-padrão.
- Mínimo e máximo.
- Percentis 25, 75, 90, 95 e 99.

As durações serão expressas em minutos.

### 7. Investigação de outliers

Será utilizado o critério exploratório do intervalo
interquartil (IQR):

IQR = Q3 - Q1

Limite inferior = Q1 - 1,5 × IQR

Limite superior = Q3 + 1,5 × IQR

Os valores fora desses limites serão identificados e
quantificados, mas não removidos.

Outliers não são necessariamente erros. Podem representar
situações operacionais reais.

### 8. Estratégia de processamento

- Leitura do arquivo Parquet original em lotes com PyArrow.
- Processamento de apenas sete campos temporais.
- Conversões realizadas exclusivamente em memória.
- Acumulação de estatísticas e resultados de diagnóstico.
- Nenhuma gravação ou alteração no arquivo original.

### 9. Resultados esperados

A análise produzirá:

1. Perfil de qualidade dos campos temporais.
2. Distribuição dos meses presentes.
3. Consistência cronológica dos eventos.
4. Classificação temporal das coletas.
5. Estatísticas das durações operacionais.
6. Quantificação de possíveis outliers.
7. Exemplos de registros que exigem investigação.

### 10. Aplicação no Power BI

Os resultados servirão como referência para a página
de AED temporal do projeto:

`NSLAB_Logistics_Shanghai_Exploratorio.pbip`

O Power BI permitirá explorar visualmente as distribuições,
os atrasos e as inconsistências, com filtros e segmentações.

**Importante:** esta etapa não realiza ETL nem tratamento
dos dados. As decisões de transformação serão tomadas
posteriormente, com base nas evidências da AED.



---

## 11. Conceitos de Python utilizados nesta análise

Esta seção apresenta os principais recursos de Python, Pandas,
NumPy e PyArrow utilizados no script, com exemplos simplificados.

O objetivo é compreender não apenas o resultado da análise,
mas também como o código funciona.

### 11.1 — Bibliotecas e importações

```python
import re
from collections import Counter

import numpy as np
import pandas as pd
```

**Conceitos:**

- `import`: disponibiliza uma biblioteca no programa.
- `as`: define um apelido para facilitar sua utilização.
- `re`: biblioteca de expressões regulares, utilizada para validar padrões textuais.
- `Counter`: estrutura para contabilizar ocorrências.
- `numpy`: biblioteca para cálculos numéricos e estatísticos.
- `pandas`: biblioteca para manipulação e análise de dados.

**Exemplo:**

```python
from collections import Counter

situacoes = ["Atrasada", "No prazo", "Atrasada"]

contagem = Counter(situacoes)

print(contagem)
```

Resultado:

```text
Counter({'Atrasada': 2, 'No prazo': 1})
```

Na nossa AED, utilizamos `Counter` para acumular quantidades
de registros inválidos, atrasados e outras classificações.

### 11.2 — Variáveis, constantes e listas

```python
ANO_REFERENCIA = 2022
TAMANHO_LOTE = 20_000

CAMPOS_DATA = [
    "accept_time",
    "pickup_time"
]
```

**Conceitos:**

- Variáveis armazenam valores utilizados durante a execução.
- Nomes em letras maiúsculas indicam, por convenção, configurações que não pretendemos alterar.
- Listas (`list`) armazenam coleções ordenadas de elementos.
- O caractere `_` pode separar milhares em números Python.

Por exemplo, `20_000` é equivalente a `20000`.

A lista `CAMPOS_DATA` permite aplicar a mesma validação a
várias colunas, evitando escrever código repetitivo.

### 11.3 — Dicionários e estruturas de acumulação

```python
qualidade = {
    "accept_time": Counter(),
    "pickup_time": Counter()
}
```

**Conceito:**

Um dicionário (`dict`) armazena pares de chave e valor.

No exemplo, cada campo possui um contador independente.

```python
qualidade["accept_time"]["nulos"] += 3
```

Essa instrução incrementa em três a quantidade de nulos
identificados no campo `accept_time`.

No script, utilizamos dicionários para organizar os resultados
por campo e por tipo de verificação.

### 11.4 — Expressões regulares (Regex)

```python
PADRAO = re.compile(
    r"^(0[1-9]|1[0-2])-"
    r"(0[1-9]|[12][0-9]|3[01]) "
    r"([01][0-9]|2[0-3]):"
    r"[0-5][0-9]:[0-5][0-9]$"
)
```

**Conceito:**

Uma expressão regular descreve um padrão que determinado
texto deve seguir.

Nosso padrão verifica o formato:

`MM-DD HH:MM:SS`

Alguns elementos:

| Expressão | Significado |
|---|---|
| `^` | Início do texto |
| `$` | Fim do texto |
| `[0-9]` | Um dígito entre 0 e 9 |
| `0[1-9]` | Valores de 01 a 09 |
| `1[0-2]` | Valores de 10 a 12 |
| `\|` | Alternativa entre padrões |

**Exemplo:**

```python
texto = "07-08 10:10:00"

resultado = PADRAO.fullmatch(texto)

print(resultado is not None)
```

Resultado:

```text
True
```

**Importante:** Regex valida o formato, mas não garante que
a data exista no calendário.

Por exemplo, `02-30 10:00:00` pode respeitar o padrão textual,
embora 30 de fevereiro não seja uma data válida.

Por isso, o script realiza também a conversão e validação
com Pandas.

### 11.5 — Leitura em lotes com PyArrow

```python
for lote in parquet.iter_batches(
    batch_size=20_000,
    columns=COLUNAS
):
    df = lote.to_pandas()
```

**Conceitos:**

- `for`: estrutura de repetição.
- `iter_batches()`: percorre o arquivo Parquet em lotes.
- `batch_size`: quantidade máxima de registros solicitada por lote.
- `columns`: seleciona somente as colunas necessárias.
- `to_pandas()`: converte o lote PyArrow em um DataFrame Pandas.

**Exemplo conceitual:**

Um arquivo com 100.000 registros, lido em lotes de 20.000,
seria processado em aproximadamente cinco iterações.

A vantagem é evitar carregar todas as colunas e registros
do arquivo simultaneamente.

No nosso projeto, utilizamos essa estratégia para preservar
a eficiência e a escalabilidade da análise.

### 11.6 — DataFrame e Series do Pandas

```python
df = lote.to_pandas()

serie = df["pickup_time"]
```

**Conceitos:**

- `DataFrame`: estrutura tabular com linhas e colunas.
- `Series`: estrutura unidimensional, normalmente uma coluna.
- `df["pickup_time"]`: seleciona uma coluna do DataFrame.

**Exemplo:**

```python
df_exemplo = pd.DataFrame({
    "order_id": [1, 2, 3],
    "pickup_time": [
        "07-08 10:10:00",
        "07-08 11:20:00",
        None
    ]
})
```

O Pandas permite aplicar operações sobre uma coluna inteira,
sem precisar escrever um `for` para cada registro.

Esse processo é chamado de operação vetorizada.

### 11.7 — Valores nulos e máscaras booleanas

```python
nulos = serie.isna()
preenchidos = ~nulos
```

**Conceitos:**

- `isna()`: identifica valores ausentes.
- `~`: inverte valores booleanos.
- `&`: operador lógico E, elemento a elemento.
- `|`: operador lógico OU, elemento a elemento.

**Exemplo:**

```python
valores = pd.Series([10, None, 30])

print(valores.isna().tolist())
```

Resultado:

```text
[False, True, False]
```

Uma máscara booleana é uma sequência de `True` e `False`
utilizada para selecionar ou contabilizar registros.

**Exemplo:**

```python
numeros = pd.Series([10, 20, 30])

mascara = numeros > 15

print(numeros[mascara].tolist())
```

Resultado:

```text
[20, 30]
```

Na nossa análise, máscaras identificam registros com datas
inválidas, janelas invertidas e coletas atrasadas.

### 11.8 — Conversão de strings para datetime

```python
convertido = pd.to_datetime(
    texto_com_ano,
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)
```

**Conceitos:**

- `pd.to_datetime()`: converte textos em datas e horários.
- `format`: especifica o formato esperado.
- `errors="coerce"`: converte valores impossíveis de interpretar em `NaT`.
- `NaT`: representação de data ou horário ausente no Pandas.

Símbolos utilizados:

| Símbolo | Significado |
|---|---|
| `%Y` | Ano com quatro dígitos |
| `%m` | Mês |
| `%d` | Dia |
| `%H` | Hora, de 00 a 23 |
| `%M` | Minuto |
| `%S` | Segundo |

**Exemplo:**

```python
datas = pd.Series([
    "2022-07-08 10:10:00",
    "2022-02-30 10:10:00"
])

resultado = pd.to_datetime(
    datas,
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print(resultado)
```

A primeira data será convertida corretamente.

A segunda será representada como `NaT`, pois 30 de fevereiro
não existe.

Na AED, essa conversão é realizada somente em memória.
Os valores originais do Parquet permanecem intactos.

### 11.9 — Funções definidas com def

```python
def atualizar_extremos(campo, serie):
    validos = serie.dropna()

    if validos.empty:
        return

    minimo = validos.min()
    maximo = validos.max()
```

**Conceitos:**

- `def`: declara uma função.
- Parâmetros: valores recebidos pela função.
- `if`: executa uma ação quando uma condição é satisfeita.
- `return`: encerra a função ou devolve um resultado.
- `dropna()`: desconsidera valores ausentes na operação.
- `min()` e `max()`: calculam extremos.

No script completo, essa função também compara os extremos
do lote atual com os extremos encontrados anteriormente.

Criar funções evita duplicação de código e facilita testes
e manutenção.

### 11.10 — Cálculo de intervalos temporais

```python
duracao = (
    coleta - aceite
).dt.total_seconds() / 60
```

**Conceitos:**

A subtração entre dois valores `datetime` produz um intervalo
de tempo (`timedelta`).

O método `total_seconds()` converte esse intervalo em segundos.

A divisão por 60 transforma o resultado em minutos.

**Exemplo:**

- Aceite: 08:13
- Coleta: 10:10
- Duração: 117 minutos

No dataset, essa métrica representa o tempo transcorrido
entre o aceite e a coleta, não necessariamente um atraso.

Para calcular atraso, precisamos comparar a coleta com o
fim da janela prevista.

### 11.11 — Classificação utilizando condições lógicas

```python
antecipada = coleta < inicio

dentro = (
    (coleta >= inicio)
    & (coleta <= fim)
)

atrasada = coleta > fim
```

**Conceito:**

Comparações entre colunas produzem máscaras booleanas.

O Pandas avalia essas condições para todos os registros
do lote.

No script completo, também exigimos datas válidas e
uma janela temporal não invertida antes de classificar.

**Exemplo:**

Janela prevista: 09:00 às 11:00.

| Horário da coleta | Classificação |
|---|---|
| 08:50 | Antecipada |
| 10:10 | Dentro da janela |
| 11:30 | Atrasada |

Essas classificações são diagnósticas. A interpretação
final do SLA dependerá das regras de negócio.

### 11.12 — Estatísticas descritivas com NumPy

```python
q1, mediana, q3, p90, p95, p99 = np.percentile(
    valores,
    [25, 50, 75, 90, 95, 99]
)
```

**Conceitos:**

- Média: soma dos valores dividida pela quantidade.
- Mediana: valor central da distribuição.
- Desvio-padrão: medida de dispersão dos valores.
- Percentil: ponto abaixo do qual se encontra determinada
  proporção das observações.

**Exemplo:**

Durações em minutos:

`[10, 20, 30, 40, 100]`

- Média: 40 minutos.
- Mediana: 30 minutos.
- Máximo: 100 minutos.

A diferença entre média e mediana pode indicar influência
de valores extremos.

O percentil 95 (P95) permite investigar a duração abaixo
da qual se encontram aproximadamente 95% das observações.

### 11.13 — Identificação de outliers pelo IQR

```python
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
```

**Conceito:**

O intervalo interquartil (IQR) mede a distância entre
o primeiro quartil (Q1) e o terceiro quartil (Q3).

**Exemplo:**

Considere:

- Q1 = 20 minutos
- Q3 = 60 minutos
- IQR = 40 minutos

Os limites serão:

- Inferior: 20 - 1,5 × 40 = -40 minutos.
- Superior: 60 + 1,5 × 40 = 120 minutos.

Uma duração de 180 minutos seria identificada como
possível outlier superior.

Isso não significa que a duração esteja incorreta.

Na AED, apenas identificaremos e quantificaremos esses
registros, preservando todos os dados.

### 11.14 — Acumulação dos resultados

```python
duracoes[nome].append(valores)
```

**Conceito:**

`append()` adiciona um elemento ao final de uma lista.

No script, armazenamos os arrays de duração produzidos
em cada lote.

Posteriormente:

```python
valores = np.concatenate(partes)
```

`np.concatenate()` combina os arrays em uma única sequência,
permitindo calcular percentis e outliers sobre todos os
valores válidos da métrica.

Essa abordagem evita carregar todas as colunas originais
simultaneamente, embora mantenha as durações calculadas
em memória para a análise estatística.

### 11.15 — Consolidação dos resultados

```python
df_qualidade = pd.DataFrame(linhas_qualidade)

display(df_qualidade)
```

**Conceitos:**

- `pd.DataFrame()`: constrói uma tabela Pandas.
- `display()`: apresenta a tabela formatada no Jupyter Notebook.
- `print()`: apresenta mensagens ou valores textuais.

Os resultados consolidados permitem comparar campos,
identificar problemas e documentar evidências.

### 11.16 — Validação da própria análise

```python
total_registros == metadata.num_rows
```

**Conceito:**

O operador `==` verifica igualdade e retorna `True` ou `False`.

Essa comparação confirma se a quantidade de registros
processados corresponde à quantidade informada nos
metadados do Parquet.

Também verificamos se a soma das classificações temporais
corresponde à quantidade de registros comparáveis.

Essas verificações ajudam a detectar falhas no próprio
processamento exploratório.

### 12. Síntese dos conceitos aprendidos

| Recurso | Aplicação na AED |
|---|---|
| `list` | Organizar campos e resultados |
| `dict` | Associar campos a indicadores |
| `Counter` | Contabilizar ocorrências |
| `for` | Percorrer campos e lotes |
| `if` | Avaliar condições |
| `def` | Criar funções reutilizáveis |
| Regex | Validar padrões textuais |
| PyArrow | Ler Parquet em lotes |
| Pandas | Analisar e comparar colunas |
| Máscaras booleanas | Identificar subconjuntos de registros |
| `datetime` | Interpretar datas e horários |
| `timedelta` | Calcular durações |
| NumPy | Calcular estatísticas e percentis |
| IQR | Identificar possíveis outliers |
| `display()` | Apresentar resultados no notebook |

### Conclusão metodológica

A análise combina leitura eficiente, validação semântica,
operações vetorizadas, estatística descritiva e verificações
de consistência.

A finalidade é compreender o comportamento dos dados
antes de qualquer transformação.

Os resultados orientarão a página de AED no Power BI
e, posteriormente, as decisões de ETL.
